# Setup

In [1]:
# Imports and project paths
from pathlib import Path
import sqlite3
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import math
from requests.exceptions import ChunkedEncodingError, RequestException
import time

# Project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "db" / "app.db"
SCHEMA_PATH = PROJECT_ROOT / "db" / "schema.sql"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH exists:", DB_PATH.exists(), DB_PATH)
print("SCHEMA_PATH exists:", SCHEMA_PATH.exists(), SCHEMA_PATH)

PROJECT_ROOT: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation
DB_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db
SCHEMA_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/schema.sql


In [2]:
# import os

# if DB_PATH.exists():
#     os.remove(DB_PATH)
#     print("Deleted existing DB")
# else:
#     print("No DB to delete")

# python scripts/init_db.py

# Config

In [3]:
# PubMed API configuration
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

NCBI_TOOL = "diffusion_topic_evolution"
NCBI_EMAIL = "REMOVED_EMAIL"
NCBI_API_KEY = None  # add later if needed

# Query
PUBMED_QUERY = '("covid-19"[Title/Abstract]) AND 2020[pdat]'

# Scaling parameters
TARGET_PMIDS = 1000
SEARCH_PAGE_SIZE = 200
FETCH_BATCH_SIZE = 25
SLEEP_SECONDS = 0.8

print("Query:", PUBMED_QUERY)
print("TARGET_PMIDS:", TARGET_PMIDS)
print("SEARCH_PAGE_SIZE:", SEARCH_PAGE_SIZE)
print("FETCH_BATCH_SIZE:", FETCH_BATCH_SIZE)
print("SLEEP_SECONDS:", SLEEP_SECONDS)

Query: ("covid-19"[Title/Abstract]) AND 2020[pdat]
TARGET_PMIDS: 1000
SEARCH_PAGE_SIZE: 200
FETCH_BATCH_SIZE: 25
SLEEP_SECONDS: 0.8


# Search

In [4]:
all_pmids = []
retstart = 0

while len(all_pmids) < TARGET_PMIDS:
    page_retmax = min(SEARCH_PAGE_SIZE, TARGET_PMIDS - len(all_pmids))

    search_params = {
        "db": "pubmed",
        "term": PUBMED_QUERY,
        "retstart": retstart,
        "retmax": page_retmax,
        "retmode": "json",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        search_params["api_key"] = NCBI_API_KEY

    search_url = f"{EUTILS_BASE}/esearch.fcgi"
    search_resp = requests.get(search_url, params=search_params, timeout=30)
    search_resp.raise_for_status()

    search_data = search_resp.json()
    batch_pmids = search_data["esearchresult"].get("idlist", [])
    total_count = int(search_data["esearchresult"].get("count", "0"))

    if not batch_pmids:
        print("No more PMIDs returned; stopping early.")
        break

    all_pmids.extend(batch_pmids)
    all_pmids = list(dict.fromkeys(all_pmids))  # preserve order, drop duplicates

    print(
        f"retstart={retstart:4d} | fetched={len(batch_pmids):3d} | "
        f"collected={len(all_pmids):4d} / target={TARGET_PMIDS} | total_available={total_count}"
    )

    retstart += len(batch_pmids)
    time.sleep(SLEEP_SECONDS)

print("\nFinal PMID count collected:", len(all_pmids))
print("First 10 PMIDs:", all_pmids[:10])
print("Last 10 PMIDs:", all_pmids[-10:])

retstart=   0 | fetched=200 | collected= 200 / target=1000 | total_available=80733
retstart= 200 | fetched=200 | collected= 400 / target=1000 | total_available=80733
retstart= 400 | fetched=200 | collected= 600 / target=1000 | total_available=80733
retstart= 600 | fetched=200 | collected= 800 / target=1000 | total_available=80733
retstart= 800 | fetched=200 | collected=1000 / target=1000 | total_available=80733

Final PMID count collected: 1000
First 10 PMIDs: ['39931522', '39911268', '39022319', '39022317', '38046875', '38046874', '38046872', '38046869', '38046867', '37928102']
Last 10 PMIDs: ['34173618', '34173616', '34173615', '34173614', '34173611', '34173610', '34173609', '34173607', '34173606', '34173605']


# Fetch

In [5]:
def fetch_pubmed_batch(pmids, max_retries=4, base_sleep=1.0):
    """
    Fetch a batch of PubMed records (XML) with retry logic.
    """

    fetch_params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "tool": NCBI_TOOL,
        "email": NCBI_EMAIL,
    }

    if NCBI_API_KEY:
        fetch_params["api_key"] = NCBI_API_KEY

    fetch_url = f"{EUTILS_BASE}/efetch.fcgi"

    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(fetch_url, params=fetch_params, timeout=60)
            resp.raise_for_status()
            return resp.text
        except (ChunkedEncodingError, RequestException) as e:
            print(f"  attempt {attempt}/{max_retries} failed: {type(e).__name__} - {e}")
            if attempt == max_retries:
                raise
            time.sleep(base_sleep * attempt)

    raise RuntimeError("fetch_pubmed_batch failed unexpectedly")

# Parse

In [6]:
# PubMed XML parsing helpers
def safe_text(elem):
    return elem.text.strip() if elem is not None and elem.text is not None else None


def extract_abstract_text(article):
    abstract_nodes = article.findall(".//Abstract/AbstractText")
    if not abstract_nodes:
        return None

    parts = []
    for node in abstract_nodes:
        label = node.attrib.get("Label")
        text = "".join(node.itertext()).strip()
        if text:
            parts.append(f"{label}: {text}" if label else text)

    return " ".join(parts) if parts else None


def extract_journal_pub_date(article):
    pub_date = article.find(".//JournalIssue/PubDate")
    if pub_date is None:
        return None, None

    year = safe_text(pub_date.find("Year"))
    month = safe_text(pub_date.find("Month"))
    day = safe_text(pub_date.find("Day"))

    publication_date = "-".join([x for x in [year, month, day] if x])
    publication_year = int(year) if year and year.isdigit() else None

    return publication_date or None, publication_year


def extract_article_date(article):
    article_date = article.find(".//Article/ArticleDate")
    if article_date is None:
        return None, None

    year = safe_text(article_date.find("Year"))
    month = safe_text(article_date.find("Month"))
    day = safe_text(article_date.find("Day"))

    article_date_str = "-".join([x for x in [year, month, day] if x])
    article_year = int(year) if year and year.isdigit() else None

    return article_date_str or None, article_year


def parse_pubmed_xml_to_records(xml_text):
    root = ET.fromstring(xml_text)
    articles = root.findall(".//PubmedArticle")

    records = []
    for article in articles:
        pmid = safe_text(article.find(".//PMID"))
        title_node = article.find(".//ArticleTitle")
        title = "".join(title_node.itertext()).strip() if title_node is not None else None
        abstract = extract_abstract_text(article)
        journal = safe_text(article.find(".//Journal/Title"))

        journal_pub_date, journal_pub_year = extract_journal_pub_date(article)
        article_date, article_year = extract_article_date(article)

        # canonical project time field:
        # prefer journal publication year/date, then fall back to article date
        publication_year = journal_pub_year if journal_pub_year is not None else article_year
        publication_date = journal_pub_date if journal_pub_date is not None else article_date

        records.append({
            "source": "pubmed",
            "source_doc_id": pmid,
            "title": title,
            "abstract": abstract,
            "publication_year": publication_year,
            "publication_date": publication_date,
            "journal": journal,
            "article_date": article_date,
            "article_year": article_year,
            "journal_pub_date": journal_pub_date,
            "journal_pub_year": journal_pub_year,
        })

    return records

In [7]:
all_records = []

n_batches = math.ceil(len(all_pmids) / FETCH_BATCH_SIZE)
print("Number of fetch batches:", n_batches)

for i in range(0, len(all_pmids), FETCH_BATCH_SIZE):
    batch_num = i // FETCH_BATCH_SIZE + 1
    pmid_batch = all_pmids[i:i + FETCH_BATCH_SIZE]

    print(f"Fetching batch {batch_num}/{n_batches} with {len(pmid_batch)} PMIDs...")
    xml_text = fetch_pubmed_batch(pmid_batch)
    batch_records = parse_pubmed_xml_to_records(xml_text)
    all_records.extend(batch_records)

    time.sleep(SLEEP_SECONDS)

print("Total parsed records:", len(all_records))

Number of fetch batches: 40
Fetching batch 1/40 with 25 PMIDs...
Fetching batch 2/40 with 25 PMIDs...
Fetching batch 3/40 with 25 PMIDs...
Fetching batch 4/40 with 25 PMIDs...
Fetching batch 5/40 with 25 PMIDs...
Fetching batch 6/40 with 25 PMIDs...
Fetching batch 7/40 with 25 PMIDs...
Fetching batch 8/40 with 25 PMIDs...
Fetching batch 9/40 with 25 PMIDs...
Fetching batch 10/40 with 25 PMIDs...
Fetching batch 11/40 with 25 PMIDs...
Fetching batch 12/40 with 25 PMIDs...
Fetching batch 13/40 with 25 PMIDs...
Fetching batch 14/40 with 25 PMIDs...
Fetching batch 15/40 with 25 PMIDs...
Fetching batch 16/40 with 25 PMIDs...
Fetching batch 17/40 with 25 PMIDs...
Fetching batch 18/40 with 25 PMIDs...
Fetching batch 19/40 with 25 PMIDs...
Fetching batch 20/40 with 25 PMIDs...
Fetching batch 21/40 with 25 PMIDs...
Fetching batch 22/40 with 25 PMIDs...
Fetching batch 23/40 with 25 PMIDs...
Fetching batch 24/40 with 25 PMIDs...
Fetching batch 25/40 with 25 PMIDs...
Fetching batch 26/40 with 25 PM

# Clean

In [8]:
df_raw = pd.DataFrame(all_records)

print("Raw batch shape:", df_raw.shape)

print("\nMissing values:")
display(df_raw.isna().sum())

print("\nCanonical publication_year counts:")
display(df_raw["publication_year"].value_counts(dropna=False).sort_index())

display(df_raw.head())

Raw batch shape: (993, 11)

Missing values:


source                0
source_doc_id         0
title                 0
abstract            255
publication_year      0
publication_date      0
journal               0
article_date        141
article_year        141
journal_pub_date      0
journal_pub_year      0
dtype: int64


Canonical publication_year counts:


publication_year
2020    586
2021    364
2022     36
2023      7
Name: count, dtype: int64

,source,source_doc_id,title,abstract,publication_year,publication_date,journal,article_date,article_year,journal_pub_date,journal_pub_year
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research,2024-12-16,2024.0,2020,2020
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research,2024-07-29,2024.0,2020,2020
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020
3,pubmed,39022317,When worlds collide: Food allergy and the COVI...,NaN,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020
4,pubmed,38046875,Should critically ill patients with COVID-19 b...,NaN,2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020


In [10]:
# Clean parsed PubMed records
df = df_raw.copy()

# Keep only rows with usable text and time fields
df = df.dropna(subset=["title", "abstract", "publication_year"])

# Standardize text-like fields
for col in [
    "title",
    "abstract",
    "journal",
    "publication_date",
    "article_date",
    "journal_pub_date",
]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Remove rows where missing values were cast to the literal string "nan"
df = df[
    (df["title"].str.len() > 0) &
    (df["abstract"].str.len() > 0) &
    (df["title"].str.lower() != "nan") &
    (df["abstract"].str.lower() != "nan")
].copy()

# Remove duplicates within the batch
df = df.drop_duplicates(subset=["source", "source_doc_id"])

# Create text field for downstream embeddings
df["clean_text"] = df["title"] + " " + df["abstract"]

print("Cleaned batch shape:", df.shape)

print("\nMissing values after cleaning:")
display(df.isna().sum())

print("\nCleaned publication_year counts:")
display(df["publication_year"].value_counts(dropna=False).sort_index())

display(df.head())

Cleaned batch shape: (738, 12)

Missing values after cleaning:


source               0
source_doc_id        0
title                0
abstract             0
publication_year     0
publication_date     0
journal              0
article_date        89
article_year        89
journal_pub_date     0
journal_pub_year     0
clean_text           0
dtype: int64


Cleaned publication_year counts:


publication_year
2020    450
2021    261
2022     27
Name: count, dtype: int64

,source,source_doc_id,title,abstract,publication_year,publication_date,journal,article_date,article_year,journal_pub_date,journal_pub_year,clean_text
0,pubmed,39931522,Study protocol: Comparison of different risk p...,"On March 11th 2020, the World Health Organizat...",2020,2020,Wellcome open research,2024-12-16,2024.0,2020,2020,Study protocol: Comparison of different risk p...
1,pubmed,39911268,"Mental health, sleep quality and quality of li...",BACKGROUND: Since the World Health Organizatio...,2020,2020,F1000Research,2024-07-29,2024.0,2020,2020,"Mental health, sleep quality and quality of li..."
2,pubmed,39022319,Parent perspectives on food allergy management...,BACKGROUND: U.S. national emergency was declar...,2020,2020-Dec,Journal of food allergy,2020-12-01,2020.0,2020-Dec,2020,Parent perspectives on food allergy management...
5,pubmed,38046874,COVID-19 critical illness in Sweden: character...,Objective: During the coronavirus disease 2019...,2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020,COVID-19 critical illness in Sweden: character...
8,pubmed,38046867,Geolocated Twitter-based population mobility i...,"Using geotagged Twitter data in Victoria, we c...",2020,2020-Dec,Critical care and resuscitation : journal of t...,2023-10-18,2023.0,2020-Dec,2020,Geolocated Twitter-based population mobility i...


# DB Insert

In [11]:
conn = sqlite3.connect(DB_PATH)

cols = [
    "source",
    "source_doc_id",
    "title",
    "abstract",
    "publication_year",
    "publication_date",
    "journal",
    "clean_text",
    "article_date",
    "article_year",
    "journal_pub_date",
    "journal_pub_year",
]

df_to_insert = df[cols].copy()

insert_sql = """
INSERT OR IGNORE INTO documents (
    source,
    source_doc_id,
    title,
    abstract,
    publication_year,
    publication_date,
    journal,
    clean_text,
    article_date,
    article_year,
    journal_pub_date,
    journal_pub_year
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

before_n = pd.read_sql_query("SELECT COUNT(*) AS n FROM documents", conn)["n"].iloc[0]

conn.executemany(insert_sql, df_to_insert.values.tolist())
conn.commit()

after_n = pd.read_sql_query("SELECT COUNT(*) AS n FROM documents", conn)["n"].iloc[0]

print("Rows before insert:", before_n)
print("Rows attempted:", len(df_to_insert))
print("Rows after insert:", after_n)
print("Rows newly added:", after_n - before_n)

conn.close()

Rows before insert: 0
Rows attempted: 738
Rows after insert: 738
Rows newly added: 738


# Validation

In [12]:
conn = sqlite3.connect(DB_PATH)

year_counts = pd.read_sql_query("""
    SELECT publication_year, COUNT(*) AS n_docs
    FROM documents
    GROUP BY publication_year
    ORDER BY publication_year
""", conn)

dup_df = pd.read_sql_query("""
    SELECT source, source_doc_id, COUNT(*) AS n
    FROM documents
    GROUP BY source, source_doc_id
    HAVING COUNT(*) > 1
""", conn)

print("Counts by year:")
display(year_counts)

print("\nDuplicates:")
display(dup_df)

sample_df = pd.read_sql_query("""
    SELECT
        id,
        source_doc_id,
        title,
        publication_year,
        journal
    FROM documents
    ORDER BY id
    LIMIT 10
""", conn)

print("\nSample rows:")
display(sample_df)

conn.close()

Counts by year:


,publication_year,n_docs
0,2020,450
1,2021,261
2,2022,27



Duplicates:


,source,source_doc_id,n



Sample rows:


,id,source_doc_id,title,publication_year,journal
0,1,39931522,Study protocol: Comparison of different risk p...,2020,Wellcome open research
1,2,39911268,"Mental health, sleep quality and quality of li...",2020,F1000Research
2,3,39022319,Parent perspectives on food allergy management...,2020,Journal of food allergy
3,4,38046874,COVID-19 critical illness in Sweden: character...,2020,Critical care and resuscitation : journal of t...
4,5,38046867,Geolocated Twitter-based population mobility i...,2020,Critical care and resuscitation : journal of t...
5,6,37671353,Public Involvement & Engagement in health ineq...,2020,International journal of population data science
6,7,37649990,Using data linkage to monitor COVID-19 vaccina...,2020,International journal of population data science
7,8,33117894,A living mapping review for COVID-19 funded re...,2020,Wellcome open research
8,9,36753237,[COVID-19 and its socio-cultural imagery in La...,2020,"Revista de salud publica (Bogota, Colombia)"
9,10,36753227,"[Risks, contamination and prevention against C...",2020,"Revista de salud publica (Bogota, Colombia)"
